In [1]:
import pandas as pd
from scipy.stats import wilcoxon

In [2]:
headers = ["Model", "Dataset","Variate", "Lookback", "PredLength", "Seed", "MSE", "MAE"]

DLinear = pd.read_csv("Savedresults/Results_336_1/Result_DLinear.csv", sep="-", header=None, names=headers)
iTrans = pd.read_csv("Savedresults/Results_336_1/Result_iTransformer.csv", sep="-", header=None, names=headers)
Frnet = pd.read_csv("Savedresults/Results_336_1/Result_FrNet.csv", sep="-", header=None, names=headers)
PatchTST = pd.read_csv("Savedresults/Results_336_old/Result_PatchTST.csv", sep="-", header=None, names=headers)
ModelX = pd.read_csv("Savedresults/Results_336_1/Result_HaDAM.csv", sep="-", header=None, names=headers)
#raw.head()

In [3]:
def setup(df):
    raw2 = df.copy()
    raw2["Seed"] = raw2["Seed"].apply(lambda x: x.replace("seed", ""))
    raw2["Lookback"] = raw2["Lookback"].apply(lambda x: x.replace("sl", ""))
    raw2["PredLength"] = raw2["PredLength"].apply(lambda x: x.replace("pl", ""))
    raw2["MSE"] = raw2["MSE"].apply(lambda x: x.replace("mse:", ""))
    raw2["MAE"] = raw2["MAE"].apply(lambda x: x.replace("mae:", ""))


    raw2["Seed"] = raw2["Seed"].astype(int)
    raw2["Lookback"] = raw2["Lookback"].astype(int)
    raw2["PredLength"] = raw2["PredLength"].astype(int)
    raw2["MSE"] = raw2["MSE"].astype(float)
    raw2["MAE"] = raw2["MAE"].astype(float)
    return raw2


dlinear_results = setup(DLinear)
itrans_results = setup(iTrans)
frnet_results = setup(Frnet)
patchtst_results = setup(PatchTST)
modelx_results = setup(ModelX)

In [4]:
# Mean and Standard Deviation for MSE and MAE for Lookback 336 for 5 different seeds

dlinear_tab = dlinear_results.groupby(["Dataset", "PredLength"]).agg({"MSE": ["mean", "std"], "MAE": ["mean", "std", "count"]})
itrans_tab = itrans_results.groupby(["Dataset", "PredLength"]).agg({"MSE": ["mean", "std"], "MAE": ["mean", "std", "count"]})
frnet_tab = frnet_results.groupby(["Dataset", "PredLength"]).agg({"MSE": ["mean", "std"], "MAE": ["mean", "std", "count"]})
patchtst_tab = patchtst_results.groupby(["Dataset", "PredLength"]).agg({"MSE": ["mean", "std"], "MAE": ["mean", "std", "count"]})
modelx_tab = modelx_results.groupby(["Dataset", "PredLength"]).agg({"MSE": ["mean", "std"], "MAE": ["mean", "std", "count"]})

In [5]:
patchtst_tab

MSE                 MAE                
                        mean       std      mean       std count
Dataset PredLength                                              
ELT     48          0.126555       NaN  0.226560       NaN     1
        96          0.146447       NaN  0.248096       NaN     1
        192         0.167131       NaN  0.260923       NaN     1
ETTh1   48          0.376436  0.003448  0.421281  0.003734     5
        96          0.472858  0.006044  0.474976  0.005438     5
        192         0.499907  0.006947  0.499854  0.004309     5
        336         0.545187  0.011950  0.531521  0.007423     5
        512         0.600907  0.021336  0.570888  0.010081     5
        720         0.814454  0.016604  0.672083  0.008597     5
ETTh2   48          0.136748  0.001608  0.253728  0.001854     5
        96          0.226959  0.004175  0.325978  0.002817     5
        192         0.202301  0.004567  0.312072  0.003578     5
        336         0.229502  0.007555  0.341005  0.005175     5
        512         0.301249  0.007987  0.389178  0.005907     5
        720         0.356460  0.010064  0.419097  0.008158     5
ETTm1   48          0.325347  0.011552  0.365451  0.004888     5
        96          0.368133  0.002037  0.393408  0.001749     5
        192         0.397297  0.010075  0.424263  0.003677     5
        336         0.427335  0.006802  0.449840  0.004562     5
        512         0.439038  0.006355  0.468793  0.002851     5
        720         0.479586  0.008863  0.495260  0.004928     5
ETTm2   48          0.090686  0.001308  0.199242  0.001286     5
        96          0.145364  0.003162  0.251305  0.002112     5
        192         0.150046  0.002753  0.262574  0.002405     5
        336         0.187778  0.007361  0.298619  0.006332     5
        512         0.198499  0.005121  0.309630  0.003398     5
        720         0.243552  0.006814  0.340010  0.002964     5
WTH     48          0.111844  0.001172  0.160239  0.001049     5
        96          0.143717  0.002004  0.217296  0.002348     5
        192         0.187728  0.001429  0.240812  0.001913     5
        336         0.273118  0.003681  0.300508  0.002108     5
        512         0.297711  0.003450  0.311026  0.002670     5
        720         0.349163  0.002129  0.361364  0.000867     5

In [ ]:
import pandas as pd
import glob
import os

# path to your files
path = "/Users/adityadey/Documents/MyProjects/dataset/tx-pv-2006/5/*.csv"
files = sorted(glob.glob(path))

dfs = []

for i, file in enumerate(files):
    df = pd.read_csv(file)

    # parse datetime (important: specify format to avoid ambiguity)
    df['LocalTime'] = pd.to_datetime(df['LocalTime'], format="%m/%d/%y %H:%M")

    # rename Power column to S1, S2, ...
    df = df.rename(columns={"Power(MW)": f"S{i+1}"})

    # set index for alignment
    df = df.set_index("LocalTime")

    dfs.append(df)

# combine all dataframes on time index
final_df = pd.concat(dfs, axis=1)

# optional: sort index (just to be safe)
final_df = final_df.sort_index()

# save to csv
final_df.to_csv("combined_power.csv")

print(final_df.head())

In [ ]:
final_df 

In [ ]:
import numpy as np

data = np.load("/Users/adityadey/Downloads/PEMS03.npz")

In [ ]:
data

In [ ]:
for key in data.files:
    print(key, data[key].shape)

In [ ]:
arr = data["data"]   # key name you saw
print(arr.shape)     # (26208, 358, 1)
arr = arr.squeeze(-1)
arr

In [ ]:
import pandas as pd

df = pd.DataFrame(arr)

In [ ]:
df.shape

In [ ]:
import pandas as pd

df = pd.read_hdf("/Users/adityadey/Downloads/metr-la.h5")


In [ ]:
import h5py
import pandas as pd

In [ ]:
f = h5py.File("/Users/adityadey/Downloads/metr-la.h5", "r")
print(list(f.keys()))

In [ ]:
data = f["df"]
print(data)

In [ ]:
print(list(f["df"].keys()))

In [ ]:
group = f["df"]

data = group["block0_values"][:]   # actual values
columns = group["axis0"][:]        # sensor IDs
index = group["axis1"][:]          # timestamps

# decode bytes → string if needed
columns = [c.decode() if isinstance(c, bytes) else c for c in columns]
index = [i.decode() if isinstance(i, bytes) else i for i in index]

df = pd.DataFrame(data, index=index, columns=columns)

print(df.shape)
df.index = pd.to_datetime(df.index.astype("int64"), unit="ns")
print(df.head())
df


In [ ]:
df.to_csv("pems-bay.csv")